# 原 39 项输入经过完整 f696 提案链的敏感度

按冻结训练中心值只中性化主输入，逐项和按三组重放模型及 EMA/HOLD。结果只反映旧模型路径依赖，不构成新因子或可交易账户。

In [ ]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
import hashlib,json
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
from src.workflow.minute_pit_allocator import _frozen_predictions
from src.workflow.pit_constituent_auxiliary_training_allocation import PitAuxiliaryAllocationSpec,AuxiliaryTrainingCandidate,_smooth_weights

root=Path.cwd();card=root/"research/stage1_f696_input_path_sensitivity_20261024"
plan_path=card/"plan.json";plan=json.loads(plan_path.read_text());dep=plan["dependencies"]
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan["frozen"] and all(v==0 for k,v in plan["budget"].items() if k!="ablation_replays_max")
assert plan["budget"]["ablation_replays_max"]==42
bridge=root/dep["bridge_card"];interval=root/dep["interval_card"]
assert sha(bridge/"plan.json")==dep["bridge_plan_sha256"] and sha(bridge/"result.json")==dep["bridge_result_sha256"]
assert sha(interval/"plan.json")==dep["interval_plan_sha256"] and sha(interval/"result.json")==dep["interval_result_sha256"]
assert sha(root/"src/workflow/minute_pit_allocator.py")==dep["forward_code_sha256"]
bridge_plan=json.loads((bridge/"plan.json").read_text());br=json.loads((bridge/"result.json").read_text())
interval_plan=json.loads((interval/"plan.json").read_text());iv=json.loads((interval/"result.json").read_text())
assert br["identity_gate_pass"] and br["sessions"]==944 and iv["rolling126_windows"]==819
src=bridge_plan["sources"];action=root/src["action_card"];parent=root/src["parent_card"];f696=root/src["f696_card"]
for path,key in [(action/"summary.json","action_summary_sha256"),(parent/"summary.json","parent_summary_sha256"),(f696/"summary.json","f696_summary_sha256")]:assert sha(path)==src[key]
sa=json.loads((action/"summary.json").read_text());sp=json.loads((parent/"summary.json").read_text());sf=json.loads((f696/"summary.json").read_text())
assert sa["nas_evidence"]["sha256"]==src["action_archive_sha256"] and sa["nas_evidence"]["size_bytes"]==src["action_archive_size_bytes"]
assert sp["nas_evidence"]["sha256"]==src["parent_archive_sha256"] and sp["nas_evidence"]["size_bytes"]==src["parent_archive_size_bytes"]
assert sf["nas_evidence"]["sha256"]==src["f696_archive_sha256"] and sf["nas_evidence"]["size_bytes"]==src["f696_archive_size_bytes"]
ar=action/"qlib_recorder"/src["action_experiment_id"]/src["action_recorder_id"]/"artifacts"
pr=parent/"qlib_recorder"/src["parent_experiment_id"]/src["parent_recorder_id"]/"artifacts"
fr=f696/"qlib_recorder"/src["f696_experiment_id"]/src["f696_recorder_id"]/"artifacts"
for name,digest in src["action_artifacts"].items():assert sha(ar/name)==digest,name
for name,digest in src["parent_artifacts"].items():assert sha(pr/name)==digest,name
assert sha(fr/"inputs/frame")==src["f696_frame_sha256"]
assert sha(root/src["provider"]/"qlib_store_manifest.json")==src["provider_manifest_sha256"]
qlib.init(provider_uri=str((root/src["provider"]).resolve()),region="us",expression_cache=None,dataset_cache=None)
with R.uri_context(str(action/"qlib_recorder")):
 rec=R.get_recorder(experiment_id=src["action_experiment_id"],recorder_id=src["action_recorder_id"])
 action_objects={name:rec.load_object(name) for name in src["action_artifacts"]}
with R.uri_context(str(parent/"qlib_recorder")):
 rec=R.get_recorder(experiment_id=src["parent_experiment_id"],recorder_id=src["parent_recorder_id"])
 parent_objects={name:rec.load_object(name) for name in src["parent_artifacts"]}
with R.uri_context(str(f696/"qlib_recorder")):
 rec=R.get_recorder(experiment_id=src["f696_experiment_id"],recorder_id=src["f696_recorder_id"])
 frame=rec.load_object("inputs/frame")
market=action_objects["inputs/market"]
dates=pd.DatetimeIndex(market.signal_date)
assert len(dates)==944 and dates.equals(pd.DatetimeIndex(frame.signal_date))
original=sp["input_audit"]["original_features"]
assert len(original)==39
features=parent_objects["inputs/frozen_parent_models"]
state=features["outer_refit"]
assert list(state["primary_features"])==original and len(state["primary_scaler"]["mean"])>=39
parent_plan=parent_objects["inputs/plan"]
base_inputs={"frozen_models":features,"features":None,"common":None,"frame":None}
with R.uri_context(str(parent/"qlib_recorder")):
 rec=R.get_recorder(experiment_id=src["parent_experiment_id"],recorder_id=src["parent_recorder_id"])
 base_inputs["features"]=rec.load_object("inputs/features")
 base_inputs["common"]=rec.load_object("inputs/common")
 base_inputs["frame"]=rec.load_object("inputs/daily_frame")
assert dates.isin(base_inputs["features"].index).all()
assert list(base_inputs["features"].columns[:39])==original
raw=action_objects["inputs/raw_predictions"]["frozen_current"].reindex(dates)
target=action_objects["inputs/source_targets"]["frozen_current"].reindex(dates)
columns=["soxl_weight","soxx_weight","cash_weight","tail_probability","alpha_probability"]
pred=_frozen_predictions(base_inputs,window="outer",plan=parent_plan).reindex(dates)[columns]
raw_parity={m:float((pred[m]-raw[m]).abs().max()) for m in columns}
assert max(raw_parity.values())<=1e-7,raw_parity
spec=PitAuxiliaryAllocationSpec(root=root,smoothing_span=parent_plan["smoothing_span"],rebalance_threshold=parent_plan["rebalance_threshold"])
candidate=AuxiliaryTrainingCandidate(**state["candidate"])
returns=market.set_index("signal_date")[["soxl_return","soxx_return"]]
assert returns.index.equals(dates)
accepted=_smooth_weights(pred,spec=spec,candidate=candidate,asset_returns=returns)
frame_outputs=frame.set_index(pd.DatetimeIndex(frame.signal_date))[columns]
target_parity={m:float(max((accepted[m]-target[m]).abs().max(),(target[m]-frame_outputs[m]).abs().max())) for m in columns}
assert max(target_parity.values())<=1e-7,target_parity
calendar=pd.DatetimeIndex(D.calendar(start_time="2011-01-03",end_time="2026-10-09",freq="day"))
assert base_inputs["features"].index.equals(calendar)
family={name:original[start:end+1] for name,(start,end) in plan["method"]["families_by_original_feature_index"].items()}
assert [x for group in family.values() for x in group]==original
runs=[];family_delta={}
for kind,label,names in [("single",name,[name]) for name in original]+[("family",name,names) for name,names in family.items()]:
 changed=base_inputs["features"].copy()
 for feature in names:
  centre=float(state["primary_scaler"]["mean"][feature])
  assert np.isfinite(centre)
  changed.loc[dates,feature]=centre
 inputs={**base_inputs,"features":changed}
 changed_raw=_frozen_predictions(inputs,window="outer",plan=parent_plan).reindex(dates)[columns]
 changed_target=_smooth_weights(changed_raw,spec=spec,candidate=candidate,asset_returns=returns)
 delta=changed_target[columns[:3]]-accepted[columns[:3]]
 assert np.isfinite(delta.to_numpy(float)).all()
 entry={"kind":kind,"name":label,"feature_count":len(names),"mean_signed_cash_delta":float(delta.cash_weight.mean()),
        "mean_abs_cash_delta":float(delta.cash_weight.abs().mean()),"max_abs_cash_delta":float(delta.cash_weight.abs().max()),
        "mean_abs_soxl_delta":float(delta.soxl_weight.abs().mean()),"mean_abs_soxx_delta":float(delta.soxx_weight.abs().mean())}
 if kind=="family":
  entry["annual_mean_signed_cash_delta"]={str(year):float(delta.loc[dates.year==year,"cash_weight"].mean()) for year in (2023,2024,2025,2026)}
  family_delta[label]=delta.cash_weight.reset_index(drop=True)
 runs.append(entry)
assert len(runs)==42 and len(family_delta)==3
account_src=interval_plan["source"]
account=root/account_src["card"]
account_summary=json.loads((account/"summary.json").read_text())
assert account_summary["nas_readback"]["verified"] and account_summary["nas_readback"]["sha256"]==account_src["nas_verified_sha256"]
account_artifacts=account/"qlib_recorder"/account_src["experiment_id"]/account_src["recorder_id"]/"artifacts"
for name,digest in account_src["artifact_sha256"].items():assert sha(account_artifacts/name)==digest,name
with R.uri_context(str(account/"qlib_recorder")):
 rec=R.get_recorder(experiment_id=account_src["experiment_id"],recorder_id=account_src["recorder_id"])
 daily=rec.load_object("analysis/daily").sort_values("trade_date").reset_index(drop=True)
 rolling=rec.load_object("analysis/rolling126")
 periods=rec.load_object("analysis/periods")
assert len(daily)==944 and len(rolling)==819 and len(periods.loc[periods.kind.eq("quarter")])==16
assert pd.DatetimeIndex(daily.signal_date).equals(dates)
close=D.features(["SOXX"],["$close"],start_time="2011-01-03",end_time="2026-10-09",freq="day")["$close"].droplevel("instrument").sort_index()
assert close.index.equals(calendar)
win=[]
for i in range(819):
 rw=rolling.iloc[i];segment=daily.iloc[i:i+126]
 assert pd.Timestamp(segment.trade_date.iloc[0])==pd.Timestamp(rw.first_trade)
 signal=pd.Timestamp(segment.signal_date.iloc[0]);pos=calendar.get_loc(signal);assert pos>=20
 r5=float(close.iloc[pos]/close.iloc[pos-5]-1);r20=float(close.iloc[pos]/close.iloc[pos-20]-1)
 state_name="uptrend" if r5>=0 and r20>=0 else "rebound" if r5>=0 else "pullback" if r20>=0 else "decline"
 row={"year":int(pd.Timestamp(rw.first_trade).year),"state":state_name,"outcome":"weak" if rw.relative_log_excess<0 else "good" if rw.relative_log_excess>0 else "tie"}
 row.update({name:float(delta.iloc[i:i+126].mean()) for name,delta in family_delta.items()})
 win.append(row)
win=pd.DataFrame(win)
assert win.outcome.value_counts().to_dict()=={"weak":564,"good":255}
cells=[]
for (state_name,year),g in win.groupby(["state","year"],sort=True):
 a=g[g.outcome.eq("weak")];b=g[g.outcome.eq("good")]
 c={"state":state_name,"year":int(year),"weak_windows":len(a),"good_windows":len(b),"matched":bool(len(a) and len(b))}
 if c["matched"]:c["weak_minus_good_family_cash_delta"]={name:float(a[name].mean()-b[name].mean()) for name in family}
 cells.append(c)
matched=[c for c in cells if c["matched"]]
assert len(matched)==12 and all(c["good_windows"]==0 for c in cells if c["year"]==2023)
w=np.array([min(c["weak_windows"],c["good_windows"]) for c in matched],dtype=float)
matched_weighted={name:float(np.average([c["weak_minus_good_family_cash_delta"][name] for c in matched],weights=w)) for name in family}
matched_signs={name:{"positive_cells":sum(c["weak_minus_good_family_cash_delta"][name]>0 for c in matched),
                    "negative_cells":sum(c["weak_minus_good_family_cash_delta"][name]<0 for c in matched)} for name in family}
result={"status":"read_only_original39_primary_path_sensitivity_no_factor_selected",
        "plan_sha256":sha(plan_path),"parent_recorder_id":src["parent_recorder_id"],"action_recorder_id":src["action_recorder_id"],
        "f696_recorder_id":src["f696_recorder_id"],"account_recorder_id":account_src["recorder_id"],
        "baseline_raw_parity":raw_parity,"baseline_target_parity":target_parity,"held_sessions":944,"rolling126_windows":819,
        "ablation_replays":len(runs),"runs":runs,"matched_cells":len(matched),"matched_family_weak_minus_good_cash_delta":matched_weighted,
        "matched_family_signs":matched_signs,"cells":cells,
        "new_market_requests":0,"new_model_fits":0,"new_native_accounts":0,"new_bootstraps":0,"new_recorders":0,
        "feature_selected":False,"policy_inference":False,"independent_validation":False,
        "limits":["Training-mean neutralization breaks joint feature relationships and is a model path-dependence probe only",
                  "Separate common-input state is held fixed even where it describes related market information",
                  "All weak/good windows are overlapping reused history; no selection, fit or funded account is permitted from this audit"]}
(card/"result.json").write_text(json.dumps(result,ensure_ascii=False,indent=2)+"\n")
print({"raw_parity":raw_parity,"target_parity":target_parity,"replays":len(runs),"families":matched_weighted})


## 中文结论

原计划只读封存模型与 f696 来源，固定39个单项和3个家族遮蔽重放，0拟合、0新账户。代码已通过裸预测和 EMA/HOLD 对原提案的身份门，并完成42次重放；随后在读取旧区间卡的来源字段时引用不存在的 `nas_verified_sha256`，于区间分组和 `result.json` 写入之前停止。随后静态检查发现预定的 `artifact_sha256` 字段也不在该旧计划中。这是依赖清单接口写错，**不是因子检验失败**。

42次只读预算已耗尽，中间逐因子数值没有保存；不重复这批重放，不把未完成的区间分析写成结果。保留原冻结计划与 notebook 源码及具体失败位置；本卡决定 `not_evaluated_no_factor_or_interval_result`。下一个机制需在调用任何模型前完成全部来源字段与日历预检，并具有独立的经济假说；SOXX 目标仍未达成。
